# Per-article BCQuality coverage (code-review)

Ad-hoc analysis of how the code-review gold dataset maps onto BCQuality
knowledge articles. Every finding is annotated with the article it derives
from (`ReviewComment.article`), and false-positive-guard entries carry their
association at entry level (`metadata.articles`). This notebook aggregates
those annotations via `bcbench.dataset.coverage`.

Set `BCQUALITY_ROOT` (or edit the cell below) to point at a BCQuality checkout
to also surface articles with **zero** gold coverage; without it the report
covers declared articles only.

In [ ]:
import os

from bcbench.dataset import CodeReviewEntry
from bcbench.dataset.coverage import build_coverage_report, enumerate_inventory, resolve_bcquality_root
from bcbench.types import EvaluationCategory

entries = CodeReviewEntry.load(EvaluationCategory.CODE_REVIEW.dataset_path)

root = resolve_bcquality_root(os.environ.get("BCQUALITY_ROOT"))
inventory = enumerate_inventory(root) if root is not None else None

report = build_coverage_report(entries, inventory)
summary = f"{len(report.covered)} articles covered across {report.annotated_entries}/{report.total_entries} annotated entries"
if report.inventory_available:
    summary += f"; {len(report.zero_coverage)} of {report.inventory_size} articles have zero coverage"
print(summary)

## Coverage by domain

In [ ]:
import pandas as pd

domains = sorted({c.domain for c in report.covered} | {a.split("/", 1)[0] for a in report.zero_coverage})
rows = []
for domain in domains:
    covered = [c for c in report.covered if c.domain == domain]
    zero = [a for a in report.zero_coverage if a.split("/", 1)[0] == domain]
    row = {"Domain": domain, "Covered": len(covered), "Gold entries": sum(c.count for c in covered)}
    if report.inventory_available:
        row["Inventory"] = len(covered) + len(zero)
        row["Zero-cov"] = len(zero)
    rows.append(row)

coverage_df = pd.DataFrame(rows)
print(coverage_df.to_string(index=False))

## Covered articles and the entries that exercise them

In [ ]:
article_df = pd.DataFrame(
    [{"Article": c.article, "Entries": c.count, "Instance IDs": ", ".join(c.entry_ids)} for c in report.covered]
).sort_values(["Article"]) if report.covered else pd.DataFrame(columns=["Article", "Entries", "Instance IDs"])
print(article_df.to_string(index=False))

## Gaps: unknown slugs, zero-coverage articles, unannotated entries

In [ ]:
if report.unknown_articles:
    print(f"Unknown article slugs not in inventory ({len(report.unknown_articles)}):")
    for a in report.unknown_articles:
        print(f"  - {a}")

if report.zero_coverage:
    print(f"\nZero-coverage articles ({len(report.zero_coverage)}):")
    for a in report.zero_coverage:
        print(f"  - {a}")

if report.unannotated_entry_ids:
    print(f"\nUnannotated entries ({len(report.unannotated_entry_ids)}):")
    for e in report.unannotated_entry_ids:
        print(f"  - {e}")